In [ ]:
#you must run this from within the terminal before continuing!
#/Applications/Google\ Chrome.app/Contents/MacOS/Google\ Chrome --remote-debugging-port=9222 --no-first-run --no-default-browser-check --user-data-dir=$(mktemp -d -t 'chrome-remote_data_dir')




In [1]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from requests import Session
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import sys
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.select import Select
import pandas as pd
from selenium.webdriver.support.wait import WebDriverWait 
from selenium.webdriver.support import expected_conditions as EC
from html.parser import HTMLParser
import numpy as np
import sqlite3
import math
from datetime import datetime








#from selenium import webdriver


#sys.path.append('/Applications/Google Chrome.app/Contents/MacOS')
#print(sys.path)
#import cookielib

import asyncio
from typing import List
from pyppeteer import launch

In [2]:
#you are not comfertable having passwords here, so after this log in
chrome_options = Options()
chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
#Change chrome driver path accordingly
chrome_driver = "/Applications/Google\ Chrome.app/Contents/MacOS/Google\ Chrome"
driver = webdriver.Chrome(chrome_driver, chrome_options=chrome_options)
driver.get('https://sws-gateway.streetsmartcentral.com/ui/host/?clientid=sscentral&redirecturi=https://www.streetsmartcentral.com/login/SSOGateway.aspx')

#username = "test"
#password = "test"

#uname = driver.find_element("id", "loginIdInput") 
#uname.send_keys(username)

#uname = driver.find_element("id", "passwordInput") 
#uname.send_keys(password)
#HTML = driver.page_source
#HTML

#driver.find_element("id", "btnLogin").click()
#print (driver.title)

TypeError: WebDriver.__init__() got an unexpected keyword argument 'chrome_options'

In [9]:

chrome_options = Options()
chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")


#Change chrome driver path accordingly
chrome_driver = "/Applications/Google\ Chrome.app/Contents/MacOS/Google\ Chrome"
driver = webdriver.Chrome(chrome_driver, chrome_options=chrome_options)

#you might have to add this to a new item
#driver.find_element("name","main")
element = WebDriverWait(driver, 10).until(lambda x: x.find_element("name","main")) 
driver.switch_to.frame(element)

TypeError: WebDriver.__init__() got an unexpected keyword argument 'chrome_options'

In [4]:
def repoint_browser():
    chrome_options = Options()
    chrome_options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")


    #Change chrome driver path accordingly
    chrome_driver = "/Applications/Google\ Chrome.app/Contents/MacOS/Google\ Chrome"
    driver = webdriver.Chrome(chrome_driver, chrome_options=chrome_options)

    #you might have to add this to a new item
    #driver.find_element("name","main")
    element = WebDriverWait(driver, 10).until(lambda x: x.find_element("name","main")) 
    driver.switch_to.frame(element)


In [5]:
def add_data_to_db(symbol,table,pulled_data,db):
    
    
    #db = "/Users/keirdaniels/Documents/personal Finance/trading/database/futures_data.db"
    con = sqlite3.connect(db)
    max_date_time = None

    try:
        
        pulled_data.to_sql(table, con, if_exists='fail', index=False )
        print(symbol+": this is NEW!!!! :)")
       
    except:
        
        print("already exists need to append")
        
        df = pd.read_sql_query("select * from "+table, con)
        
        try:
            df['date_time'] = pd.to_datetime(df['date_time'], format='%m/%d/%Y\n%I:%M:%S %p')
        except:
            df['date_time'] = pd.to_datetime(df['date_time'], format='%Y/%m/%d %H:%M:%S')
        
        max_date_time = df.max().date_time
        
        try:
            pulled_data['date_time_compare'] = pd.to_datetime(pulled_data['date_time'], format='%m/%d/%Y\n%I:%M:%S %p')
        except:
            pulled_data['date_time_compare'] = pd.to_datetime(pulled_data['date_time'], format='%Y/%m/%d %H:%M:%S')
        
        pulled_data = pulled_data[pulled_data.date_time_compare > max_date_time]#max_date_time]
        pulled_data = pulled_data.iloc[:,:-1]
        pulled_data.to_sql(table, con, if_exists='append',index=False)
        
        
       # dtype={"date":"TEXT",
       #                                                                    "time":"TEXT",
       #                                                                    "high":"REAL",
       #                                                                    "low":"REAL",
       #                                                                    "open":"REAL",
       #                                                                    "close":"REAL",
       #                                                                    "volume":"REAL",
       #                                                                    "date_time":"TEXT",
       #                                                                    "symbol":"REAL"}"""

   

    con.close()
    return max_date_time,pulled_data

In [6]:


def pull_options(symbol, run ):
    
    
    print("pulling this symbol: "+symbol)
    
    quotes = WebDriverWait(driver, 10).until(lambda x: x.find_element("id","quotes")) 
    webdriver.common.action_chains.ActionChains(driver).move_to_element(quotes).perform()

    Option_chains = driver.find_element(By.PARTIAL_LINK_TEXT,"Futures Options")
    webdriver.common.action_chains.ActionChains(driver).move_to_element(quotes).click(Option_chains).perform()

    quote_symbol = WebDriverWait(driver, 10).until(lambda x: x.find_element("id","txtSymbol")) 
    quote_symbol.clear()
    quote_symbol.send_keys(symbol)

    Range =  Select(driver.find_element("id", 'lstRange'))
    Range.select_by_value("0")

    Expiration =  Select(driver.find_element("id", 'lstMonths'))
    Expiration.select_by_visible_text("All")

    time_now = time.strftime("%Y-%m-%d %H:%M:%S ", time.localtime())
    button = driver.find_element(By.NAME,"cmdSubmit").click()

    count  = 0
    while WebDriverWait(driver, 10).until(EC.text_to_be_present_in_element((By.ID, 'tblQuotebox'),symbol)) ==False:
        count += 1
        if count > 1000:
            print("sybol did not update")
            break

    
   
    table  = WebDriverWait(driver, 10).until(lambda x: x.find_element("id", 'tblQuotebox')) 
    table.get_attribute('outerHTML')
    future_data = pd.read_html(table.get_attribute('outerHTML'),header=0)
    futures_info = future_data[0]
    
    table  = WebDriverWait(driver, 10).until(lambda x: x.find_element("id", 'tbChain'))
    option_data = pd.read_html(table.get_attribute('outerHTML'),header=0)
    option_data = option_data[0]
    
    return option_data,futures_info,time_now


In [7]:
def add_fututes_info_columns(option_table, futures_info, time_now):
    column_name = (["symbol","last_price","date_time"])
    list_length = len(option_table)
    temp_list = [np.full(list_length,futures_info.Symbol[0]),
                 np.full(list_length,futures_info.Last[0]),
                 np.full(list_length,time_now)
                          ]

    counter = 0              
    while counter < len(column_name):      
        option_table[column_name[counter]] = temp_list[counter]
        counter += 1
    
    return(option_table)

def option_table_formatting(option_table):

    mat_dates = option_table.query('Symbol != Symbol').Last
    maturity =pd.Series()

    counter = 0

    while counter < len(mat_dates.index):
        current_index = mat_dates.index[counter]
        value = mat_dates[current_index]
        if counter == len(mat_dates.index)-1:
            next_index = len(option_table)
        else:
            next_index = mat_dates.index[counter+1]


        temp_array = pd.Series(np.full((next_index-current_index), value))

        maturity = maturity.append(temp_array)

        counter += 1
        
      
    option_table["maturity"] = maturity.to_list()
    option_table.drop(mat_dates.index,axis=0, inplace= True)
    
    option_table = option_table[['Symbol', 'Last', 'Chg', 'Bid', 'Ask', 'Vol' , 'OpInt', 'Strike',
       'Symbol.1' , 'Last.1' , 'Chg.1' ,'Bid.1' , 'Ask.1' , 'Vol.1', 'OpInt.1' ,"maturity"]]
    
    #option_table.drop(["Action","Action.1","Unnamed: 17", "Unnamed: 18","Unnamed: 19"],axis=1, inplace= True)
    cols = {'Symbol': "call_symbol", 'Last': "call_last_price", 'Chg': "call_chg_in_price", 'Bid': "call_bid", 'Ask': "call_ask", 'Vol' : "call_vol", 
 'OpInt' : "call_OpInt", 'Strike': "call_strike" ,
       'Symbol.1' : "put_symbol", 'Last.1' : "put_last_price", 'Chg.1' : "put_chg_in_price",
 'Bid.1' : "put_bid", 'Ask.1' : "put_ask", 'Vol.1' : "put_vol", 'OpInt.1' : "put_OpInt",
       }


    option_table.rename(columns=cols,inplace=True)

    
    return option_table


In [27]:


#pull options
symbol = ['CLV23','CLX23','CLZ23','CLF24','CLG24','CLH24',
         'CLJ24','CLK24','CLM24']
#symbol = ['CLJ24','CLK24','CLM24']
db = "/Users/keirdaniels/Documents/personal Finance/trading/database/options_data.db"
for current in symbol:
    #PULLS DATA FROM SITE
    option_table,futures_info,time_now = pull_options(current, 0)
    option_table = option_table_formatting(option_table)
    option_table = add_fututes_info_columns(option_table, futures_info, time_now)
    
    
    #Write to db
    table_option = current+"_data"
    add_data_to_db(current,table_option,option_table,db)

   

pulling this symbol: CLJ24


/var/folders/tk/b_ypk43x3xq1945h416lxdnh0000gn/T/ipykernel_81571/1906457433.py:19: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  maturity =pd.Series()
/var/folders/tk/b_ypk43x3xq1945h416lxdnh0000gn/T/ipykernel_81571/1906457433.py:34: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  maturity = maturity.append(temp_array)
/var/folders/tk/b_ypk43x3xq1945h416lxdnh0000gn/T/ipykernel_81571/1906457433.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  option_table.rename(columns=cols,inplace=True)
/var/folders/tk/b_ypk43x3xq1945h416lxdnh0000gn/T/ipykernel_81571/1287758845.py:24: FutureWarning: The default v

already exists need to append
pulling this symbol: CLK24


/var/folders/tk/b_ypk43x3xq1945h416lxdnh0000gn/T/ipykernel_81571/1906457433.py:19: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  maturity =pd.Series()
/var/folders/tk/b_ypk43x3xq1945h416lxdnh0000gn/T/ipykernel_81571/1906457433.py:34: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  maturity = maturity.append(temp_array)
/var/folders/tk/b_ypk43x3xq1945h416lxdnh0000gn/T/ipykernel_81571/1906457433.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  option_table.rename(columns=cols,inplace=True)
/var/folders/tk/b_ypk43x3xq1945h416lxdnh0000gn/T/ipykernel_81571/1287758845.py:24: FutureWarning: The default v

already exists need to append
pulling this symbol: CLM24
already exists need to append


/var/folders/tk/b_ypk43x3xq1945h416lxdnh0000gn/T/ipykernel_81571/1906457433.py:19: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  maturity =pd.Series()
/var/folders/tk/b_ypk43x3xq1945h416lxdnh0000gn/T/ipykernel_81571/1906457433.py:34: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  maturity = maturity.append(temp_array)
/var/folders/tk/b_ypk43x3xq1945h416lxdnh0000gn/T/ipykernel_81571/1906457433.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  option_table.rename(columns=cols,inplace=True)
/var/folders/tk/b_ypk43x3xq1945h416lxdnh0000gn/T/ipykernel_81571/1287758845.py:24: FutureWarning: The default v

In [17]:
def format_orders_pull(table):
    row_list = table.find_element(By.TAG_NAME, "tbody").find_elements(By.TAG_NAME, "tr")
    columns =[]
    row_count = 0
    partial_dict = {}
    
    while row_count < len(row_list):
        value = row_list[row_count].find_elements(By.TAG_NAME, "td")
        if not(len(columns)):
            columns = {"order_number": [value[3].text],
                           "symbol": [value[4].text],
                           "desc": [value[5].text],
                           "action": [value[8].text],
                           "Qty": [value[9].text],
                           "type":[value[10].text],
                           "Dur": [value[11].text],
                           "fill_price": [value[13].text],
                           "date_time": [value[14].text],
                           "create_time": [value[15].text]
                           #you will want to add infor for status if Partial,  
                          } #note! date_time = fill_time
        else:
            columns["order_number"].append(value[3].text)
            columns["symbol"].append(value[4].text)
            columns["desc"].append(value[5].text)
            columns["action"].append(value[8].text)
            columns["Qty"].append(value[9].text)
            columns["type"].append(value[10].text)
            columns["Dur"].append(value[11].text)
            columns["fill_price"].append(value[13].text)
            columns["date_time"].append(value[14].text)
            columns["create_time"].append(value[15].text) 


        
        
        if len(value[16].text) > 0: #checks for part fills and ajdust qty and comment

            try:
                parital_dict[columns[order_number][-1]].append(int(value[16].text.strip("()").split('/')[0]))


            except: 

                partial_dict.update({columns["order_number"][-1]:[int(value[16].text.strip("()").split('/')[0])]})


            if len(partial_dict[columns["order_number"][-1]]) > 1:
                diff_qty = partial_dict[columns["order_number"][-1]][-1] - partial_dict[columns["order_number"][-1]][-2]
                columns["Qty"][-1] = diff_qty
            
            else:
                diff_qty = partial_dict[columns["order_number"][-1]][-1]
                columns["Qty"][-1] = diff_qty

            #add the comment for part fills
            """ try: 
                columns["comment"].append("part fill!")
                comment_added = True
            except: 
                columns.update({"comment": ["part fill!"]})
                comment_added = True """

        
        if row_count < len(row_list) -1 : #section for adding comments on non part cases
            comment_added = False
            for next_row_value in row_list[row_count + 1].find_elements(By.TAG_NAME, "td"):
                if next_row_value.get_attribute("class") == '' and row_count == 0 and "message"  in next_row_value :

                    columns.update({"comment": [next_row_value.text]})
                    row_count = row_count +1
                    comment_added = True

                elif next_row_value.get_attribute("class") == '' and "message"  in next_row_value :
                     columns["comment"].append(next_row_value.text)
                     comment_added = True

            if comment_added == False:

                if row_count == 0:
                    columns.update({"comment":['']})
                

                else:
                    columns["comment"].append('')
                    
            

        if row_count  ==  len(row_list) -1:
            columns["comment"].append('')

        row_count = row_count +1
    
  
            
    orders_table = pd.DataFrame.from_dict(columns)

    
    return orders_table



In [18]:
def pull_order_data(time_range = "13"): ##13 is month, 5 is 7d, 17 is two d, 1 is yesterday, 0 is today
    
    trade = WebDriverWait(driver, 10).until(lambda x: x.find_element("id","trade"))
    webdriver.common.action_chains.ActionChains(driver).move_to_element(trade).perform()

    order_status = driver.find_element(By.PARTIAL_LINK_TEXT,"Order Status")
    webdriver.common.action_chains.ActionChains(driver).move_to_element(trade).click(order_status).perform()


    #pull time now
    time_checker = datetime.now() #grab time stamp to confrim new table is up later"""

    orders_table_range = WebDriverWait(driver, 10).until(lambda x: x.find_element("id","lstDateRange"))
    Select(orders_table_range).select_by_value(time_range)
    Select(driver.find_element("id","lstStatusType")).select_by_value("1")
    Select(driver.find_element("id","lstSecType")).select_by_value("0")


    table_time_elm = WebDriverWait(driver, 10).until(lambda x: x.find_element("id", 'currpagetime'))
    page_time = datetime.strptime(table_time_elm.text[0:-3]
                                               , '%m/%d/%Y %I:%M:%S %p')

    count = 0
    while count < 1000:
        time.sleep(2)
        print("request time: ",time_checker)
        print(page_time)

        if page_time < time_checker:
            #reset table inputs
            print("need to update page")
            orders_table_range = WebDriverWait(driver, 10).until(lambda x: x.find_element("id","lstDateRange"))
            Select(orders_table_range).select_by_value(time_range)
            Select(driver.find_element("id","lstStatusType")).select_by_value("1")
            Select(driver.find_element("id","lstSecType")).select_by_value("0")
            driver.find_element("id","masthead").find_element(By.XPATH,"//a[@title='Refresh']").click()

            #pull time again 
            table_time_elm = WebDriverWait(driver, 10).until(lambda x: x.find_element("id", 'currpagetime'))
            page_time = datetime.strptime(table_time_elm.text[0:-3]
                                                       , '%m/%d/%Y %I:%M:%S %p')

        else:
            print("page is updated")
            break

        count = count + 1


    table = WebDriverWait(driver, 10).until(lambda x: x.find_element("id", 'mainTable'))

    
    return table

In [20]:
#repoint_browser()
#pull new order data from CH
order_data = pull_order_data()
order_data_for_db =  format_orders_pull(order_data)

#add orders data to db
db = "/Users/keirdaniels/Documents/personal Finance/trading/database/trades_data.db"
 
max_date_time,uploaded_data   = add_data_to_db("Orders","orders_data",order_data_for_db,db)
uploaded_data

request time:  2023-09-11 23:30:27.104940
2023-09-11 23:30:12
need to update page
request time:  2023-09-11 23:30:27.104940
2023-09-11 23:30:12
need to update page
request time:  2023-09-11 23:30:27.104940
2023-09-11 23:30:12
need to update page
request time:  2023-09-11 23:30:27.104940
2023-09-11 23:30:27
need to update page
request time:  2023-09-11 23:30:27.104940
2023-09-11 23:30:34
page is updated
already exists need to append


,order_number,symbol,desc,action,Qty,type,Dur,fill_price,date_time,create_time,comment
0,351034751,CLV23,Light Sweet Crude Oil October 2023,BUY,1,Limit 87.24,DAY,87.24,9/11/2023\n2:44:39 PM,9/11/2023\n2:39:03 PM,
1,351032376,CLV23,Light Sweet Crude Oil October 2023,SELL,1,Limit 87.29,DAY,87.29,9/11/2023\n2:36:21 PM,9/11/2023\n2:36:20 PM,
2,351040380,CLV23,Light Sweet Crude Oil October 2023,BUY,1,Limit 87.12,DAY,87.12,9/11/2023\n2:18:30 PM,9/11/2023\n2:18:27 PM,
3,351034301,CLV23,Light Sweet Crude Oil October 2023,SELL,1,Limit 87.17,DAY,87.17,9/11/2023\n2:17:04 PM,9/11/2023\n2:16:21 PM,
4,351035248,CLV23,Light Sweet Crude Oil October 2023,BUY,1,Limit 87.39,DAY,87.39,9/11/2023\n1:49:33 PM,9/11/2023\n12:22:58 PM,
5,351040304,CLV23,Light Sweet Crude Oil October 2023,SELL,1,Limit 87.49,DAY,87.49,9/11/2023\n12:15:23 PM,9/11/2023\n12:15:15 PM,
6,351042725,CLV23,Light Sweet Crude Oil October 2023,BUY,1,Limit 87.32,DAY,87.32,9/11/2023\n11:10:16 AM,9/11/2023\n11:08:54 AM,
7,351032240,CLV23,Light Sweet Crude Oil October 2023,SELL,1,Market,DAY,87.37,9/11/2023\n10:55:58 AM,9/11/2023\n10:55:56 AM,
8,350689648,CLV23,Light Sweet Crude Oil October 2023,BUY,1,Limit 87.54,DAY,87.49,9/11/2023\n10:02:35 AM,9/11/2023\n10:02:31 AM,
9,351034153,CLV23,Light Sweet Crude Oil October 2023,SELL,1,Market,DAY,87.63,9/11/2023\n9:53:49 AM,9/11/2023\n9:53:48 AM,
